# RNN 古诗生成 V2 —— 改进版七言绝句

以 **"明月"** 为起始词，基于增强 LSTM 网络生成七言绝句。

**改进点**
- 3层 LSTM + LayerNorm + 多层FC解码头（含激活函数）+ 残差连接
- 激活函数可选：`relu` / `gelu` / `tanh`
- 学习率调度：Warmup + CosineAnnealing
- 生成策略可选：温度采样 / Top-k 采样 / Top-p（nucleus）采样

**使用方法**：将本 Notebook 与 `data_files/` 目录（含四个JSON）放在同一目录，逐格运行。

## 0. 安装依赖（首次运行时取消注释）

In [1]:
# !pip install torch matplotlib

## 1. 导入库 & 全局配置

In [ ]:
import json, os, re, random, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ════════════════════════════════════════════════════
# 全局配置（所有超参集中于此，按需修改）
# ════════════════════════════════════════════════════
CONFIG = {
    # ── 数据 ──────────────────────────────────────
    "data_files": [
        "poet.song.40000.json",
        "poet.song.41000.json",
        "poet.song.42000.json",
        "poet.song.43000.json",
    ],
    "data_dir": "data_files",   # JSON 所在子目录

    # ── 模型结构 ──────────────────────────────────
    "embedding_dim": 256,
    "hidden_size":   512,
    "num_layers":    3,          # LSTM 层数（较原版增加1层）
    "dropout":       0.3,
    "activation":    "gelu",     # FC头激活函数: "relu" / "gelu" / "tanh"

    # ── 训练超参 ──────────────────────────────────
    "batch_size":      64,
    "num_epochs":      30,
    "learning_rate":   1e-3,     # Adam 峰值学习率
    "warmup_epochs":   3,        # Warmup 轮数
    "label_smoothing": 0.1,      # Label Smoothing 系数
    "clip_grad":       5.0,
    "seed":            42,

    # ── 生成参数 ──────────────────────────────────
    "start_words":    "明月",
    "sample_mode":    "topp",    # "temperature" / "topk" / "topp"
    "temperature":    0.8,
    "top_k":          10,        # top-k 采样保留的候选数
    "top_p":          0.9,       # nucleus 采样的累积概率阈值

    # ── 输出路径 ──────────────────────────────────
    "save_model":  "poem_lstm_v2.pth",
    "loss_fig":    "training_loss_v2.png",
}

# ── 随机种子 & 设备 ──────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

PAD, START, END, UNK = "<PAD>", "<START>", "<END>", "<UNK>"


Using device: cuda


## 2. 数据加载与预处理

数据集中七言绝句格式：**2 个 paragraph，每个 16 字**
```
"XXXXXXX，XXXXXXX。"
```
过滤后展开为长度 32 的字符序列。

In [3]:
_QIYAN_PATTERN = re.compile(
    r'^[\u4e00-\u9fff]{7}[，,][\u4e00-\u9fff]{7}[。！？]$'
)

def is_qiyan_jueju(paragraphs):
    if len(paragraphs) != 2:
        return False
    for p in paragraphs:
        ps = p.strip()
        if len(ps) != 16 or not _QIYAN_PATTERN.match(ps):
            return False
    return True


def load_sequences(data_dir, filenames):
    sequences = []
    for fname in filenames:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"[warning] 文件不存在，跳过: {fpath}")
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            para = item.get("paragraphs", [])
            if is_qiyan_jueju(para):
                seq = "".join(p.strip() for p in para)
                sequences.append(seq)
    print(f"过滤后七言绝句: {len(sequences)} 首")
    assert len(sequences) > 0, "未找到七言绝句！请检查 data_dir 配置。"
    return sequences


def build_vocab(sequences):
    chars = sorted(set("".join(sequences)))
    vocab = [PAD, START, END, UNK] + chars
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"词表大小: {len(vocab)}")
    return char2idx, idx2char, vocab


sequences            = load_sequences(CONFIG["data_dir"], CONFIG["data_files"])
char2idx, idx2char, vocab = build_vocab(sequences)

print("\n示例序列：")
for s in sequences[:3]:
    print(" ", s)


[warning] 文件不存在，跳过: data_files\poet_song_40000.json
[warning] 文件不存在，跳过: data_files\poet_song_41000.json
[warning] 文件不存在，跳过: data_files\poet_song_42000.json
[warning] 文件不存在，跳过: data_files\poet_song_43000.json
过滤后七言绝句: 0 首


AssertionError: 未找到七言绝句！请检查 data_dir 配置。

## 3. 构建 PyTorch Dataset

In [ ]:
class PoemDataset(Dataset):
    """
    Teacher Forcing 样本对，序列长度 32：
        inp = [START] + seq[:-1]
        tgt = seq
    """
    def __init__(self, sequences, char2idx):
        start_id = char2idx[START]
        unk_id   = char2idx[UNK]
        self.data = []
        for seq in sequences:
            ids = [char2idx.get(c, unk_id) for c in seq]
            self.data.append((
                torch.tensor([start_id] + ids[:-1], dtype=torch.long),
                torch.tensor(ids, dtype=torch.long),
            ))

    def __len__(self):         return len(self.data)
    def __getitem__(self, i):  return self.data[i]


dataset = PoemDataset(sequences, char2idx)
print(f"训练样本数: {len(dataset)}")


## 4. 改进模型定义

```
Embedding(256)
    ↓
3层 LSTM(512)  + Dropout(层间)
    ↓
LayerNorm(512)
    ↓
Dropout(0.3)
    ↓
FC(512 → 256) → Activation(GELU/ReLU/Tanh)
    ↓  ↘ Residual(线性投影后相加)
FC(256 → vocab_size)
```


In [ ]:
def get_activation(name: str):
    """根据名称返回激活函数模块"""
    name = name.lower()
    if name == "relu":  return nn.ReLU()
    if name == "gelu":  return nn.GELU()
    if name == "tanh":  return nn.Tanh()
    raise ValueError(f"不支持的激活函数: {name}，可选: relu / gelu / tanh")


class ImprovedPoemLSTM(nn.Module):
    """
    改进版字符级 LSTM 语言模型

    结构：
        Embedding
        → 3层 LSTM（层间 Dropout）
        → LayerNorm
        → Dropout
        → FC(H → H//2) + Activation          ← 多层解码头
        → FC(H//2 → vocab_size)
        （FC头之间加残差：proj(x) + fc1(x)）

    参数：
        vocab_size  : 词表大小
        emb_dim     : Embedding 维度
        hidden_size : LSTM 隐层维度 H
        num_layers  : LSTM 层数
        dropout     : Dropout 比率
        activation  : FC头激活函数名称
    """
    def __init__(self, vocab_size, emb_dim, hidden_size,
                 num_layers, dropout, activation="gelu"):
        super().__init__()

        # ── Embedding ────────────────────────────────
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        # ── 多层 LSTM ────────────────────────────────
        self.lstm = nn.LSTM(
            input_size  = emb_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )

        # ── 输出归一化 ───────────────────────────────
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout    = nn.Dropout(dropout)

        # ── 多层 FC 解码头 ───────────────────────────
        mid = hidden_size // 2
        self.fc1        = nn.Linear(hidden_size, mid)
        self.activation = get_activation(activation)
        self.fc2        = nn.Linear(mid, vocab_size)

        # 残差分支：将 LSTM 输出投影到 mid 维，与 fc1 输出相加
        self.residual_proj = nn.Linear(hidden_size, mid, bias=False)

        # ── 权重初始化 ───────────────────────────────
        for fc in [self.fc1, self.fc2, self.residual_proj]:
            nn.init.xavier_uniform_(fc.weight)
            if fc.bias is not None:
                nn.init.zeros_(fc.bias)

    def forward(self, x, hidden=None):
        """
        x      : (B, L)  整数 token id
        返回   : logits (B, L, V),  hidden
        """
        # Embedding + dropout
        emb = self.dropout(self.embedding(x))           # (B, L, E)

        # LSTM
        out, hidden = self.lstm(emb, hidden)             # (B, L, H)

        # LayerNorm + Dropout
        out = self.dropout(self.layer_norm(out))         # (B, L, H)

        # 多层 FC 解码头 + 残差
        residual = self.residual_proj(out)               # (B, L, mid)
        h        = self.activation(self.fc1(out))        # (B, L, mid)
        h        = h + residual                          # 残差相加
        logits   = self.fc2(h)                           # (B, L, V)

        return logits, hidden

    def init_hidden(self, batch, device):
        h = torch.zeros(self.lstm.num_layers, batch,
                        self.lstm.hidden_size, device=device)
        return (h, torch.zeros_like(h))


# ── 实例化 ───────────────────────────────────────────
model = ImprovedPoemLSTM(
    vocab_size  = len(vocab),
    emb_dim     = CONFIG["embedding_dim"],
    hidden_size = CONFIG["hidden_size"],
    num_layers  = CONFIG["num_layers"],
    dropout     = CONFIG["dropout"],
    activation  = CONFIG["activation"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {n_params:,}")
print(model)


## 5. 学习率调度器

使用 **Warmup + CosineAnnealing** 策略：
- 前 `warmup_epochs` 个 epoch 线性从 0 升至峰值 lr
- 之后按余弦曲线衰减至接近 0

In [ ]:
class WarmupCosineScheduler(torch.optim.lr_scheduler._LRScheduler):
    """
    前 warmup_steps 步线性升温，之后余弦退火。
    """
    def __init__(self, optimizer, warmup_epochs, total_epochs, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch  # 当前 epoch（0-based）
        lrs = []
        for base_lr in self.base_lrs:
            if ep < self.warmup_epochs:
                # 线性 warmup
                lr = base_lr * (ep + 1) / self.warmup_epochs
            else:
                # 余弦退火
                progress = (ep - self.warmup_epochs) / max(
                    1, self.total_epochs - self.warmup_epochs)
                lr = base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
            lrs.append(max(lr, 1e-6))
        return lrs

print("WarmupCosineScheduler 定义完毕。")


## 6. 训练

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, clip):
    """训练一个 epoch，返回平均 per-token loss"""
    model.train()
    total_loss, total_n = 0.0, 0
    for inp, tgt in loader:
        inp, tgt = inp.to(device), tgt.to(device)
        hidden = model.init_hidden(inp.size(0), device)

        logits, _ = model(inp, hidden)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item() * tgt.numel()
        total_n    += tgt.numel()
    return total_loss / total_n


# ── 初始化训练组件 ───────────────────────────────────
loader    = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                       shuffle=True, num_workers=0)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs = CONFIG["warmup_epochs"],
    total_epochs  = CONFIG["num_epochs"],
)
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

# ── 训练循环 ─────────────────────────────────────────
epoch_losses = []

for epoch in range(1, CONFIG["num_epochs"] + 1):
    loss = train_epoch(model, loader, optimizer, criterion,
                       DEVICE, CONFIG["clip_grad"])
    scheduler.step()
    epoch_losses.append(loss)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
          f"Loss: {loss:.4f}  LR: {lr_now:.2e}")

    if epoch % 5 == 0 or epoch == 1:
        # 在训练过程中预览（直接用临时函数，完整版见后续单元格）
        print(f"  （生成预览见第8节单元格运行后）")

print("\n训练完成！")


## 7. 保存模型

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "char2idx": char2idx,
    "idx2char":  idx2char,
    "vocab":     vocab,
    "config":    CONFIG,
}, CONFIG["save_model"])
print(f"模型已保存 → {CONFIG['save_model']}")


## 8. 绘制 Loss 收敛曲线

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
xs = list(range(1, len(epoch_losses) + 1))
ax.plot(xs, epoch_losses, "b-o", markersize=4, linewidth=1.8, label="Train Loss")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Loss",  fontsize=13)
ax.set_title("Training Loss Curve (V2 - Improved LSTM)", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(xs)
fig.tight_layout()
fig.savefig(CONFIG["loss_fig"], dpi=150)
plt.show()
print(f"Loss 曲线 → {CONFIG['loss_fig']}")


## 9. 生成函数（三种采样策略）

| 模式 | 说明 | 配置参数 |
|------|------|----------|
| `temperature` | 按温度 softmax 全量采样 | `temperature` |
| `topk` | 只在概率最高的 k 个中采样 | `top_k` |
| `topp` | 累积概率达到 p 时截断（nucleus） | `top_p` |

在 `CONFIG["sample_mode"]` 中切换。

In [ ]:
# 七言绝句标点强制位置（0-based 序列索引）
PUNCT_MAP  = {7: "，", 15: "。", 23: "，", 31: "。"}
ALL_PUNCTS = set("，。！？；、,.")


def _sample_from_logit(logit, mode, temperature, top_k, top_p):
    """
    从单步 logit (V,) 中按指定策略采样，返回 token id（int）。
    所有标点屏蔽逻辑在调用前已处理。
    """
    if mode == "topk":
        # Top-k：保留概率最高的 k 个，其余置 -inf
        values, _ = torch.topk(logit, min(top_k, logit.size(-1)))
        logit[logit < values[-1]] = -1e9
        probs = torch.softmax(logit / temperature, dim=-1)
        return torch.multinomial(probs, 1).item()

    elif mode == "topp":
        # Top-p（nucleus）：按概率从大到小累加，超过 p 则截断
        probs_sorted, sorted_idx = torch.sort(
            torch.softmax(logit / temperature, dim=-1), descending=True)
        cumsum = torch.cumsum(probs_sorted, dim=-1)
        # 移除累积概率超过 p 的 token（保留至少1个）
        sorted_idx_remove = cumsum - probs_sorted > top_p
        probs_sorted[sorted_idx_remove] = 0.0
        probs_sorted /= probs_sorted.sum()
        chosen = torch.multinomial(probs_sorted, 1).item()
        return sorted_idx[chosen].item()

    else:
        # temperature 采样（默认）
        probs = torch.softmax(logit / temperature, dim=-1)
        return torch.multinomial(probs, 1).item()


def generate(model, start_words, char2idx, idx2char, device,
             mode="topp", temperature=0.8, top_k=10, top_p=0.9,
             seq_len=32):
    """
    自回归生成一首七言绝句。

    Args:
        start_words : 起始字符串，如 "明月"
        mode        : 采样策略 "temperature" / "topk" / "topp"
        temperature : 采样温度
        top_k       : topk 采样的 k 值
        top_p       : nucleus 采样的 p 值
        seq_len     : 生成序列总长度（含标点），七言绝句固定 32
    Returns:
        格式化的 4 行诗（字符串）
    """
    model.eval()
    unk_id   = char2idx[UNK]
    start_id = char2idx[START]
    all_punct_ids = [char2idx[p] for p in ALL_PUNCTS if p in char2idx]

    generated = list(start_words)

    with torch.no_grad():
        # Step 1: 预热 hidden state
        primer = [start_id] + [char2idx.get(c, unk_id) for c in start_words]
        inp    = torch.tensor([primer], dtype=torch.long, device=device)
        hidden = model.init_hidden(1, device)
        _, hidden = model(inp, hidden)

        # Step 2: 自回归采样
        last_id = char2idx.get(start_words[-1], unk_id)
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)

        while len(generated) < seq_len:
            logits, hidden = model(inp, hidden)
            logit = logits[0, 0].clone()

            pos = len(generated)

            if pos in PUNCT_MAP:
                # 强制标点
                next_char = PUNCT_MAP[pos]
            else:
                # 屏蔽标点，按策略采样
                logit[all_punct_ids] = -1e9
                next_id = _sample_from_logit(
                    logit, mode, temperature, top_k, top_p)
                next_char = idx2char.get(next_id, UNK)

            generated.append(next_char)
            inp = torch.tensor(
                [[char2idx.get(next_char, unk_id)]],
                dtype=torch.long, device=device)

    s     = "".join(generated[:seq_len])
    lines = [s[i*8:(i+1)*8] for i in range(4)]
    return "\n".join(lines)

print("生成函数定义完毕。")


## 10. 生成古诗

In [ ]:
print("=" * 55)
print(f'以「{CONFIG["start_words"]}」为起始词生成七言绝句（{CONFIG["sample_mode"]} 采样）：')
print("=" * 55)

for i in range(5):
    poem = generate(
        model        = model,
        start_words  = CONFIG["start_words"],
        char2idx     = char2idx,
        idx2char     = idx2char,
        device       = DEVICE,
        mode         = CONFIG["sample_mode"],
        temperature  = CONFIG["temperature"],
        top_k        = CONFIG["top_k"],
        top_p        = CONFIG["top_p"],
    )
    print(f"\n【第 {i+1} 首】\n{poem}")

print("\n" + "=" * 55)


## 11. 三种采样策略对比（同一模型）

In [ ]:
modes = [
    ("temperature", dict(temperature=0.8)),
    ("topk",        dict(temperature=0.8, top_k=10)),
    ("topp",        dict(temperature=0.8, top_p=0.9)),
]

for mode_name, kwargs in modes:
    print(f"\n{'─'*40}")
    print(f"  采样策略: {mode_name}  参数: {kwargs}")
    print('─'*40)
    poem = generate(model, CONFIG["start_words"],
                    char2idx, idx2char, DEVICE,
                    mode=mode_name, **kwargs)
    print(poem)


## 12. （可选）加载已保存模型

In [ ]:
# checkpoint = torch.load(CONFIG["save_model"], map_location=DEVICE)
#
# char2idx = checkpoint["char2idx"]
# idx2char = checkpoint["idx2char"]
# vocab    = checkpoint["vocab"]
# cfg      = checkpoint["config"]
#
# model2 = ImprovedPoemLSTM(
#     vocab_size  = len(vocab),
#     emb_dim     = cfg["embedding_dim"],
#     hidden_size = cfg["hidden_size"],
#     num_layers  = cfg["num_layers"],
#     dropout     = cfg["dropout"],
#     activation  = cfg["activation"],
# ).to(DEVICE)
# model2.load_state_dict(checkpoint["model_state_dict"])
#
# poem = generate(model2, "明月", char2idx, idx2char, DEVICE, mode="topp")
# print(poem)
